<a href="https://colab.research.google.com/github/sasya025/Travclan_Section4/blob/main/mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import requests
from datetime import timedelta
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("hotel_bookings (1).csv")
df.head()

,booking_id,customer_id,customer_name,customer_segment,customer_signup_date,customer_home_city,customer_loyalty_tier,property_id,property_name,property_city,...,nights,booking_channel,adr,discount_amount,coupon_code,total_amount,payment_method,booking_status,review_rating,review_date
0,100000,424,Customer_424,Group,2023-07-03,Manali,Platinum,38,Crimson Courtyard,Manali,...,4,OTA,5808.46,0.00,NONE,46467.68,Debit Card,Cancelled,NaN,NaN
1,100001,239,Customer_239,Individual,2022-07-18,Jaipur,NaN,32,Saffron Palace,Pune,...,4,Corporate Portal,4021.62,0.00,NaN,16086.48,Net Banking,Cancelled,NaN,NaN
2,100002,301,Customer_301,Corporate,2023-07-05,Jaipur,Gold,53,Saffron Heights,Chennai,...,3,Corporate Portal,17663.11,15373.35,SAVE10,90605.31,Net Banking,Completed,NaN,NaN
3,100003,722,Customer_722,Individual,2022-11-07,Udaipur,NaN,43,Indigo Lodge,Bangalore,...,3,OTA,5885.85,0.00,NONE,17657.55,Credit Card,No-Show,NaN,NaN
4,100004,306,Customer_306,Corporate,2022-02-02,Udaipur,Silver,29,Cedar Lodge,Kochi,...,7,Direct Website,6199.44,5924.82,SAVE10,80867.34,Credit Card,Completed,6.0,2024-11-21


In [5]:
import requests
import pandas as pd

url = "https://date.nager.at/api/v3/PublicHolidays/2024/IN"

try:
    response = requests.get(url, timeout=20)

    if response.status_code == 200:

        holidays = pd.DataFrame(response.json())

        print("API call successful")
        print(f"Fetched {len(holidays)} holidays")

    else:

        print(
            f"API returned status {response.status_code}"
        )

        holidays = pd.DataFrame(
            {
                'date': [
                    '2024-01-26',
                    '2024-03-25',
                    '2024-08-15',
                    '2024-10-02',
                    '2024-12-25'
                ],
                'localName': [
                    'Republic Day',
                    'Holi',
                    'Independence Day',
                    'Gandhi Jayanti',
                    'Christmas'
                ]
            }
        )

        print("Using fallback holiday data")

except Exception as e:

    print("API Error:", e)

    holidays = pd.DataFrame(
        {
            'date': [
                '2024-01-26',
                '2024-03-25',
                '2024-08-15',
                '2024-10-02',
                '2024-12-25'
            ],
            'localName': [
                'Republic Day',
                'Holi',
                'Independence Day',
                'Gandhi Jayanti',
                'Christmas'
            ]
        }
    )

    print("Using fallback holiday data")

holidays.head()

API returned status 204
Using fallback holiday data


,date,localName
0,2024-01-26,Republic Day
1,2024-03-25,Holi
2,2024-08-15,Independence Day
3,2024-10-02,Gandhi Jayanti
4,2024-12-25,Christmas


In [6]:
holidays['date'] = pd.to_datetime(
    holidays['date']
)

holidays.head()

,date,localName
0,2024-01-26,Republic Day
1,2024-03-25,Holi
2,2024-08-15,Independence Day
3,2024-10-02,Gandhi Jayanti
4,2024-12-25,Christmas


In [7]:
from datetime import timedelta

holiday_window = set()

for holiday_date in holidays['date']:

    for i in range(-2, 3):

        holiday_window.add(
            holiday_date + timedelta(days=i)
        )

print("Total holiday-adjacent dates:", len(holiday_window))

Total holiday-adjacent dates: 25


In [8]:
df['checkin_date'] = pd.to_datetime(
    df['checkin_date']
)

df['holiday_flag'] = (
    df['checkin_date']
    .isin(holiday_window)
)

df['holiday_flag'] = df[
    'holiday_flag'
].map({
    True: 'Holiday Adjacent',
    False: 'Ordinary'
})

df[['checkin_date','holiday_flag']].head()

,checkin_date,holiday_flag
0,2024-08-30,Ordinary
1,2024-07-01,Ordinary
2,2024-02-22,Ordinary
3,2024-08-04,Ordinary
4,2024-11-11,Ordinary


In [9]:
summary = (
    df.groupby('holiday_flag')
      .agg(
          avg_booking_value=('total_amount','mean'),
          avg_nights=('nights','mean'),
          total_bookings=('booking_id','count')
      )
      .round(2)
)

summary

,avg_booking_value,avg_nights,total_bookings
holiday_flag,,,
Holiday Adjacent,29850.13,2.90,838
Ordinary,31423.34,2.85,11162


In [10]:
cancel_rate = (
    df.groupby('holiday_flag')
      ['booking_status']
      .apply(
          lambda x:
          (x=='Cancelled').mean()*100
      )
      .round(2)
)

cancel_rate

,booking_status
holiday_flag,
Holiday Adjacent,22.08
Ordinary,18.97


In [11]:
summary['cancellation_rate'] = cancel_rate

summary

,avg_booking_value,avg_nights,total_bookings,cancellation_rate
holiday_flag,,,,
Holiday Adjacent,29850.13,2.90,838,22.08
Ordinary,31423.34,2.85,11162,18.97


In [12]:
holiday = summary.loc['Holiday Adjacent']
ordinary = summary.loc['Ordinary']

booking_value_lift = (
    (holiday['avg_booking_value']
     - ordinary['avg_booking_value'])
    / ordinary['avg_booking_value']
) * 100

stay_lift = (
    (holiday['avg_nights']
     - ordinary['avg_nights'])
    / ordinary['avg_nights']
) * 100

cancel_diff = (
    holiday['cancellation_rate']
    - ordinary['cancellation_rate']
)

print("Booking Value Lift (%):", round(booking_value_lift,2))
print("Stay Length Lift (%):", round(stay_lift,2))
print("Cancellation Difference (%):", round(cancel_diff,2))

Booking Value Lift (%): -5.01
Stay Length Lift (%): 1.75
Cancellation Difference (%): 3.11
